# 04 – Combine niche model with SeaBee remote sensing

Combines the **niche model** (03) with **SeaBee remote sensing** (shallow water)
into one dataset, which is the input to 05.

- **Niche model**: `nisjemodell_norge_2026_25833.geo.parquet` – rule-based
  classification from light/salinity/substrate/intertidal.
- **SeaBee remote sensing**: `marint_32VLQ_T07_burn-all.gpkg` – observed
  habitat classes (ALGAE, ROCK, SEDIMENT, ANGIO, DEEP, ANTHRO) in shallow water.

The rules for how remote sensing overrides the niche model live in
`mnk.rules.classify_seabee` (one rule set, derived from the data model
`nkm_klassifiseringsregler`). ANGIO/ALGAE/ROCK/SEDIMENT are authoritative for
both substrate and light (`nkm_lys`).

**Performance**: the niche model is national (~1.1 M polygons). Only polygons
that actually overlap SeaBee habitat are touched – the rest pass through
unchanged without expensive geometry operations. The SeaBee tile (32VLQ) covers
only a subset.

In [1]:
import gc
from pathlib import Path

import numpy as np
import geopandas as gpd
import pandas as pd
import shapely

import mnk
import mnk.rules
import mnk.attributes
import mnk.vectorize
import mnk.utils

CRS = "EPSG:25833"
REPO = Path.cwd()

NICHE_PARQUET = REPO / "nisjemodell_norge_2026_25833.geo.parquet"
SEABEE_GPKG = REPO / "marint_32VLQ_T07_burn-all.gpkg"
# Filled tørrfall (intertidal). The 03 nisjemodell clips tørrfall against its own
# polygons, so intertidal areas with missing salinity get dropped entirely. We add
# the full tørrfall extent back here with NA classifying attributes (lys / salinitet
# / substrat / tørrfall), so they show up as "mapped, unclassified" for remote
# sensing to resolve later.
TORRFALL_PARQUET = "gs://niva-geodata/MarintNaturKart/results/torrfall-filled_norge_25833.geo.parquet"

# Habitat SeaBee classes that can override the niche model.
# DEEP/ANTHRO do not override (see mnk.rules.SEABEE_RULES).
HABITAT_CLASSES = ["ALGAE", "ROCK", "SEDIMENT", "ANGIO"]

SEABEE_METHOD = "NKM_regelbasert_fjaere"

# Codes in the niche model (nkm_*) -> tokens in mnk.rules.
def _invert(d):
    return {v: k for k, v in d.items()}

LIGHT_MAP = _invert(mnk.rules.NKM_LYS)
SALINITY_MAP = _invert(mnk.rules.NKM_SALINITET)
SUBSTRATE_MAP = _invert(mnk.rules.NKM_SUBSTRAT)
INTERTIDAL_MAP = _invert(mnk.rules.NKM_TOERFALL)

## 1. Load data

In [2]:
niche = gpd.read_parquet(NICHE_PARQUET)
assert niche.crs.to_epsg() == 25833
print(f"Niche model: {len(niche):,} polygons")

seabee = gpd.read_file(SEABEE_GPKG, layer="polygons").to_crs(CRS)
seabee_hab = seabee.loc[
    seabee["klasse"].isin(HABITAT_CLASSES), ["klasse", "geometry"]
].reset_index(drop=True)
print(f"SeaBee habitat: {len(seabee_hab):,} polygons")
print(seabee["klasse"].value_counts())

del seabee
gc.collect()

Niche model: 2,736,485 polygons


SeaBee habitat: 26,065 polygons
klasse
ALGAE       9000
ROCK        8400
ANTHRO      7581
DEEP        4718
SEDIMENT    4425
ANGIO       4240
Name: count, dtype: int64


0

## 2. Find niche model polygons overlapping SeaBee habitat

Spatial index query `sindex.query(..., predicate="intersects")` separates
**affected** polygons from those passed through unchanged. Only affected
polygons go on to the expensive intersection/difference.

In [3]:
# Spatial index query (no global union_all – scales to the whole country).
hit = niche.sindex.query(seabee_hab.geometry, predicate="intersects")
hit_idx = niche.index[np.unique(hit[1])]

affected = niche.loc[hit_idx].copy()
unaffected = niche.drop(index=hit_idx)
print(f"Affected by SeaBee habitat: {len(affected):,}  (unchanged: {len(unaffected):,})")

del niche, hit, hit_idx
gc.collect()

Affected by SeaBee habitat: 144,797  (unchanged: 2,591,688)


0

## 3. Overlay: SeaBee habitat × affected niche polygons

The intersection gives the pieces where remote sensing overrides. Then a
`difference` keeps the part of the affected polygons that lies **outside** the
habitat (unchanged niche model).

In [4]:
affected = affected.assign(
    _lys=affected[mnk.rules.COL_LYS].map(LIGHT_MAP),
    _sal=affected[mnk.rules.COL_SALINITET].map(SALINITY_MAP),
    _sub=affected[mnk.rules.COL_SUBSTRAT].map(SUBSTRATE_MAP),
    _tf=affected[mnk.rules.COL_TOERFALL].map(INTERTIDAL_MAP),
)
affected["_src"] = np.arange(len(affected))

# (a) Intersection via sjoin + vectorised shapely intersection (faster than
#     gpd.overlay, which also builds a full planar overlay).
pairs = gpd.sjoin(
    affected, seabee_hab[["klasse", "geometry"]], how="inner", predicate="intersects"
)
_rgeom = seabee_hab.geometry.to_numpy()[pairs["index_right"].to_numpy()]
_inter = shapely.intersection(pairs.geometry.to_numpy(), _rgeom)
overlaid = pairs.set_geometry(_inter, crs=CRS).drop(columns="index_right")
# Keep polygon parts only: GeometryCollections (polygon + edge line) must be
# exploded, otherwise the whole piece – and its area – is lost.
overlaid = overlaid.explode(index_parts=False)
overlaid = overlaid[overlaid.geom_type == "Polygon"]
overlaid = overlaid[~overlaid.geometry.is_empty & overlaid.geometry.notna()]

# (b) Remainder of affected polygons (outside habitat): subtract only the union of
#     pieces intersecting each source polygon (cheaper than one global union).
_clip = overlaid.dissolve(by="_src").geometry.reindex(affected["_src"].to_numpy())
_src_geom = affected.geometry.to_numpy()
_has = _clip.notna().to_numpy()
_rest = _src_geom.copy()
_rest[_has] = shapely.difference(_src_geom[_has], _clip[_has].to_numpy())
affected_rest = affected.assign(geometry=_rest)
affected_rest = affected_rest[~affected_rest.geometry.is_empty & affected_rest.geometry.notna()]
print(f"Overlay pieces: {len(overlaid):,}  | affected remainder: {len(affected_rest):,}")

del affected, seabee_hab, pairs, _rgeom, _inter, _clip, _src_geom, _has, _rest
gc.collect()

/tmp/ipykernel_22513/1350366971.py:31: UserWarning: GeoSeries.notna() previously returned False for both missing (None) and empty geometries. Now, it only returns False for missing values. Since the calling GeoSeries contains empty geometries, the result has changed compared to previous versions of GeoPandas.
Given a GeoSeries 's', you can use '~s.is_empty & s.notna()' to get back the old behaviour.

To further ignore this warning, you can do: 
import warnings; warnings.filterwarnings('ignore', 'GeoSeries.notna', UserWarning)
  affected_rest = affected_rest[~affected_rest.geometry.is_empty & affected_rest.geometry.notna()]


Overlay pieces: 257,724  | affected remainder: 112,235


0

## 4. Apply `classify_seabee` (vectorised over unique combinations)

The number of unique `(klasse, lys, salinitet, substrat, tørrfall)` combinations
is small, so the rules are resolved once per combination and merged back – no
per-polygon Python loop.

In [5]:
keys = ["klasse", "_lys", "_sal", "_sub", "_tf"]
combos = overlaid[keys].drop_duplicates().reset_index(drop=True)

def resolve(row):
    res = mnk.rules.classify_seabee(
        row["klasse"], row["_lys"], row["_sal"], row["_sub"], row["_tf"]
    )
    return pd.Series(res if res is not None else (pd.NA, pd.NA))

combos[["_nin_code", "_nin_name"]] = combos.apply(resolve, axis=1)
print(combos.to_string())

overlaid = overlaid.merge(combos, on=keys, how="left")
# Drop any non-marine pieces (classify_seabee -> None).
overlaid = overlaid[overlaid["_nin_code"].notna()].copy()

      klasse      _lys   _sal  _sub       _tf _nin_code                        _nin_name
0      ALGAE   afotisk   salt  løst   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
1   SEDIMENT   afotisk   salt  løst   utenfor   NA-MA05  Eufotisk saltvanns-sedimentbunn
2      ANGIO   afotisk   salt  løst   utenfor   NA-MB02          Saltvanns-undervannseng
3       ROCK   afotisk   salt  løst   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
4      ALGAE   afotisk   salt  fast   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
5      ANGIO   afotisk   salt  fast   utenfor   NA-MB02          Saltvanns-undervannseng
6       ROCK   afotisk   salt  fast   utenfor   NA-MA02      Eufotisk fast saltvannsbunn
7   SEDIMENT   afotisk   salt  fast   utenfor   NA-MA05  Eufotisk saltvanns-sedimentbunn
8   SEDIMENT   afotisk  brakk  fast   utenfor   NA-MC04          Brakkvanns-sedimentbunn
9      ALGAE   afotisk  brakk  fast   utenfor   NA-MC02              Fast brakkvannsbunn
10     ALGAE  eufotis

## 5. Update attributes of the remotely sensed pieces

- `nin_kode_fkid` / `NiN_name`: from the remote sensing rule.
- `nkm_prosess`: looked up from `mnk.attributes.NKM_PROSESS`.
- `nkm_lm-dk`: set to the remotely sensed substrate.
- `nkm_lys`: set from `mnk.rules.SEABEE_LYS_OVERRIDE` (bottom observed ⇒ eufotisk).
- `metode_klassifisering`: SeaBee method.
- `sannsynlighet_beskrivelse`: `medium` (remote sensing is medium-confidence).
- `datakilde`: SeaBee is authoritative for lys + substrat (`lys_fjernmaalt_seabee`,
  `substrat_fjernmaalt_seabee`); salinitet stays HI and tørrfall stays kartverket.

In [ ]:
# Remote sensing overrides the model substrate. ANGIO (undervannseng) is soft bottom.
SEABEE_SUBSTRATE_NKM = {
    "ALGAE": mnk.rules.NKM_SUBSTRAT["fast"],
    "ROCK": mnk.rules.NKM_SUBSTRAT["fast"],
    "SEDIMENT": mnk.rules.NKM_SUBSTRAT["løst"],
    "ANGIO": mnk.rules.NKM_SUBSTRAT["løst"],
}
SEABEE_LYS_NKM = {k: mnk.rules.NKM_LYS[v] for k, v in mnk.rules.SEABEE_LYS_OVERRIDE.items()}

overlaid["nin_kode_fkid"] = overlaid["_nin_code"]
overlaid["NiN_name"] = overlaid["_nin_name"]
overlaid["nkm_prosess"] = overlaid["nin_kode_fkid"].map(mnk.attributes.NKM_PROSESS)
overlaid[mnk.rules.COL_SUBSTRAT] = (
    overlaid["klasse"].map(SEABEE_SUBSTRATE_NKM).fillna(overlaid[mnk.rules.COL_SUBSTRAT])
)
overlaid[mnk.rules.COL_LYS] = (
    overlaid["klasse"].map(SEABEE_LYS_NKM).fillna(overlaid[mnk.rules.COL_LYS])
)
overlaid["metode_klassifisering"] = SEABEE_METHOD
# Remote sensing is medium-confidence, overriding the model value inherited from the
# underlying niche polygon ("medium" matches sannsynlighet_beskrivelse in 03).
overlaid["sannsynlighet_beskrivelse"] = "medium"
# SeaBee is authoritative for lys + substrat; salinitet stays HI, tørrfall stays the
# model's (kartverket — build_datakilde adds it for intertidal pieces).
overlaid["datakilde"] = mnk.attributes.build_datakilde(
    np.full(len(overlaid), mnk.attributes.SOURCE_SUBSTRAT_SEABEE),
    overlaid[mnk.rules.COL_TOERFALL],
    lys_token=mnk.attributes.SOURCE_LYS_SEABEE,
)
assert overlaid["nkm_prosess"].notna().all()

## 6. Combine

`combined = unchanged niche model + affected remainder + remotely sensed pieces`.
Only a small minority of polygons went through geometry operations.

`nkm_seabee_klasse` keeps the SeaBee observed class (ALGAE/ROCK/SEDIMENT/ANGIO) for
the remotely sensed pieces, and is NA for the pure nisjemodell pieces.

In [ ]:
# nin_int is a raster-model artefact (only needed in 03 to derive attributes). It and
# nk_fid (assigned only at the end, in 05, before the grid split) are dropped here.
# nkm_seabee_klasse keeps the SeaBee observed class; NA where there is none (pure
# nisjemodell pieces not touched by remote sensing).
unaffected["nkm_seabee_klasse"] = pd.NA
affected_rest["nkm_seabee_klasse"] = pd.NA
overlaid["nkm_seabee_klasse"] = overlaid["klasse"]

OUT_COLS = [c for c in unaffected.columns if not c.startswith("_") and c not in ("nin_int", "nk_fid")]

stamped = gpd.GeoDataFrame(
    pd.concat(
        [unaffected[OUT_COLS], affected_rest[OUT_COLS], overlaid[OUT_COLS]],
        ignore_index=True,
    ),
    crs=CRS,
)
print(f"Stamped (niche ⊕ SeaBee): {len(stamped):,} polygons")
print(stamped["nin_kode_fkid"].value_counts())

del unaffected, affected_rest, overlaid
gc.collect()

## 7. Add tørrfall (NA — awaiting remote sensing)

The niche model + SeaBee do not cover the whole intertidal zone: the 03
nisjemodell clips tørrfall against its own polygons, so intertidal areas with
**missing salinity get no polygon at all**. We add the full tørrfall extent
(`TORRFALL_PARQUET`) back here, for the part not already covered
(`torrfall − stamped`, so the classified intertidal in `stamped` is kept and the
two together preserve the whole tørrfall extent).

These added polygons carry **NA** on every classifying attribute — the four axes
(`nkm_lys`, `nkm_lm-sa`, `nkm_lm-dk`, `nkm_lm-tv`), `nin_kode_fkid` / `NiN_name`,
`nkm_prosess`, `metode_klassifisering` and `sannsynlighet_beskrivelse`. They are
*mapped but unclassified*; remote sensing or nisjemodelling fills these later.
Only `opphav_kode = NKM` is set, and the rows are identifiable by a NA
`nin_kode_fkid`.

There is **no deep-sea canvas**: open sea the model does not reach stays an
honest hole rather than being guessed as deep-salt (`NA-MA06`). Possible
brakkvannspoll are likewise not added here — they can be folded in later. The
`torrfall − stamped` difference runs tiled in `mnk.vectorize.coverage_gaps`, so
nothing is unioned/differenced globally.

In [ ]:
# Add the full tørrfall extent with NA classifying attributes -> "mapped but
# unclassified". Only opphav_kode + datakilde are set; everything classifying
# (nin_kode_fkid, NiN_name, the four axes, nkm_prosess, metode, sannsynlighet,
# nkm_seabee_klasse) stays NA until remote sensing or nisjemodelling resolves it. The
# only known source is the kartverket tørrfall layer. Rows have a NA nin_kode_fkid.
TORRFALL_DEFAULTS = {
    "opphav_kode": mnk.attributes.OPPHAV_KODE,
    "nin_kode_fkid": pd.NA,
    "NiN_name": pd.NA,
    "nkm_prosess": pd.NA,
    mnk.rules.COL_LYS: pd.NA,
    mnk.rules.COL_SALINITET: pd.NA,
    mnk.rules.COL_SUBSTRAT: pd.NA,
    mnk.rules.COL_TOERFALL: pd.NA,
    "metode_klassifisering": pd.NA,
    "sannsynlighet_beskrivelse": pd.NA,
    "datakilde": mnk.attributes.SOURCE_TOERFALL_KARTVERKET,
    "nkm_seabee_klasse": pd.NA,
}

# Only the tørrfall the nisjemodell does not already cover (torrfall - stamped),
# tiled so nothing is unioned/differenced globally. The classified intertidal
# stays in `stamped`; the rest comes in NA -> together = the full tørrfall extent.
torrfall = gpd.read_parquet(TORRFALL_PARQUET).to_crs(CRS)[["geometry"]].reset_index(drop=True)
torrfall["geometry"] = shapely.make_valid(torrfall.geometry.to_numpy())
print(f"Tørrfall: {len(torrfall):,} polygons")
tf_fill = mnk.vectorize.coverage_gaps(torrfall, stamped, tile_size=10_000.0, workers=4)
for col, val in TORRFALL_DEFAULTS.items():
    tf_fill[col] = val
del torrfall
gc.collect()

# nk_fid is assigned only in 05 (before the grid split); not set here.
final = gpd.GeoDataFrame(pd.concat([stamped, tf_fill], ignore_index=True), crs=CRS)
final = final[stamped.columns]
print(f"Coverage: {len(final):,} polygons ({len(tf_fill):,} tørrfall NA-filled)")
print(final["nin_kode_fkid"].value_counts(dropna=False))

del stamped, tf_fill
gc.collect()

## 8. Save

In [9]:
fname = mnk.utils.to_filename("nisjemodel-remote-sensing", "norge", "2026", CRS.split(":")[1])

# Parquet is cheap and memory friendly – write it first.
final.to_parquet(REPO / f"{fname}.geo.parquet", compression="snappy")
print(f"Saved {len(final):,} polygons -> {fname}.geo.parquet")

# GPKG via GDAL materialises the whole layer in memory; write in chunks (append)
# to keep memory down at national scale.
gpkg_path = REPO / f"{fname}.gpkg"
gpkg_path.unlink(missing_ok=True)
CHUNK = 100_000
for start in range(0, len(final), CHUNK):
    part = final.iloc[start : start + CHUNK]
    part.to_file(
        gpkg_path,
        driver="GPKG",
        layer="nin",
        mode="w" if start == 0 else "a",
    )
    del part
    gc.collect()
print(f"Saved {len(final):,} polygons -> {fname}.gpkg")

Saved 4,977,343 polygons -> nisjemodel-remote-sensing_norge_2026_25833.geo.parquet


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(
/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type MULTILINESTRING is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitte

/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(
/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type GEOMETRYCOLLECTION is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emi

/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(
/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type GEOMETRYCOLLECTION is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emi

/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(


... (17 outputs left out)

/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(


/home/kim/work/marint-naturkart-nivaR/.pixi/envs/default/lib/python3.13/site-packages/pyogrio/raw.py:733: RuntimeWarning: A geometry of type POLYGON is inserted into layer nin of geometry type MULTIPOLYGON, which is not normally allowed by the GeoPackage specification, but the driver will however do it. To create a conformant GeoPackage, if using ogr2ogr, the -nlt option can be used to override the layer geometry type. This warning will no longer be emitted for this combination of layer and feature geometry type.
  ogr_write(


Saved 4,977,343 polygons -> nisjemodel-remote-sensing_norge_2026_25833.gpkg
